In [2]:
import requests
import pandas as pd
import geopandas as gpd
import numpy as np

/Users/yijinwang/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Getting block groups within top 10 tracts ready

Census block groups household median income

In [3]:
API_KEY = "c9cf843d2c297d7ad549b1be51dbcf62facce09c"

ACS_URL = (
    "https://api.census.gov/data/2023/acs/acs5?"
    "get=B19013_001E,B01003_001E,NAME&"
    "for=block group:*&"
    "in=state:06+county:075&"
    f"key={API_KEY}"
)

print("Requesting ACS socioeconomic data for SF County...")

resp = requests.get(ACS_URL)
resp.raise_for_status()

data = resp.json()

# Convert to DataFrame
cols = data[0]
rows = data[1:]
acs_df = pd.DataFrame(rows, columns=cols)

# Rename
acs_df = acs_df.rename(columns={
    "B19013_001E": "median_income",
    "B01003_001E": "population"
})

# Clean numeric
acs_df["median_income"] = pd.to_numeric(acs_df["median_income"], errors="coerce")
acs_df["population"] = pd.to_numeric(acs_df["population"], errors="coerce")

# Create GEOID
acs_df["GEOID"] = acs_df["state"] + acs_df["county"] + acs_df["tract"]

# Drop bad rows
acs_df = acs_df.dropna(subset=["median_income", "population"])

print("ACS rows:", len(acs_df))
acs_df.head()


Requesting ACS socioeconomic data for SF County...
ACS rows: 681


,median_income,population,NAME,state,county,tract,block group,GEOID
0,123826,957,Block Group 1; Census Tract 101.01; San Franci...,06,075,010101,1,06075010101
1,66103,1047,Block Group 2; Census Tract 101.01; San Franci...,06,075,010101,2,06075010101
2,108417,1795,Block Group 1; Census Tract 101.02; San Franci...,06,075,010102,1,06075010102
3,189181,1310,Block Group 1; Census Tract 102.01; San Franci...,06,075,010201,1,06075010201
4,152328,1298,Block Group 2; Census Tract 102.01; San Franci...,06,075,010201,2,06075010201


10 tracts that deserve equity stations

In [4]:
top10tract = pd.read_csv(
    "top10tract.csv",
    dtype={"tractce": str}
)
top10tract.head()

,tractce,cross_count,rank
0,012002,1553,36
1,012001,1456,41
2,011102,1446,45
3,011001,1142,78
4,011101,1108,85


In [5]:
acs_df_top10tract = acs_df[acs_df["tract"].isin(top10tract["tractce"])]
print("ACS rows:", len(acs_df_top10tract))
acs_df_top10tract.head()

ACS rows: 28


,median_income,population,NAME,state,county,tract,block group,GEOID
32,87857,1173,Block Group 1; Census Tract 110.01; San Franci...,06,075,011001,1,06075011001
33,131382,1154,Block Group 2; Census Tract 110.01; San Franci...,06,075,011001,2,06075011001
37,89521,978,Block Group 1; Census Tract 111.01; San Franci...,06,075,011101,1,06075011101
38,144653,734,Block Group 2; Census Tract 111.01; San Franci...,06,075,011101,2,06075011101
39,86667,758,Block Group 3; Census Tract 111.01; San Franci...,06,075,011101,3,06075011101


Demand weight calculation

In [6]:
acs_df_top10tract = acs_df_top10tract[acs_df_top10tract['median_income'] > 0]
acs_df_top10tract['demand_weight'] = acs_df_top10tract['median_income'] * acs_df_top10tract['population']
acs_df_top10tract.head()

,median_income,population,NAME,state,county,tract,block group,GEOID,demand_weight
32,87857,1173,Block Group 1; Census Tract 110.01; San Franci...,06,075,011001,1,06075011001,103056261
33,131382,1154,Block Group 2; Census Tract 110.01; San Franci...,06,075,011001,2,06075011001,151614828
37,89521,978,Block Group 1; Census Tract 111.01; San Franci...,06,075,011101,1,06075011101,87551538
38,144653,734,Block Group 2; Census Tract 111.01; San Franci...,06,075,011101,2,06075011101,106175302
39,86667,758,Block Group 3; Census Tract 111.01; San Franci...,06,075,011101,3,06075011101,65693586


Get GEOID of bg instead of tract

In [7]:
acs_df_top10tract["GEOID_bg"] = (
    acs_df_top10tract["GEOID"] +
    acs_df_top10tract["block group"]
)
acs_df_top10tract.head()

,median_income,population,NAME,state,county,tract,block group,GEOID,demand_weight,GEOID_bg
32,87857,1173,Block Group 1; Census Tract 110.01; San Franci...,06,075,011001,1,06075011001,103056261,060750110011
33,131382,1154,Block Group 2; Census Tract 110.01; San Franci...,06,075,011001,2,06075011001,151614828,060750110012
37,89521,978,Block Group 1; Census Tract 111.01; San Franci...,06,075,011101,1,06075011101,87551538,060750111011
38,144653,734,Block Group 2; Census Tract 111.01; San Franci...,06,075,011101,2,06075011101,106175302,060750111012
39,86667,758,Block Group 3; Census Tract 111.01; San Franci...,06,075,011101,3,06075011101,65693586,060750111013


Import Block group shapefile

In [8]:
import geopandas as gpd

shp_path_bg = "tl_2023_06_bg/tl_2023_06_bg.shp"
ca_bg = gpd.read_file(shp_path_bg)

print("Total CA block groups:", len(ca_bg))

Total CA block groups: 25607


In [9]:
import pandas as pd

# 0) Make sure join keys are comparable (strings, 6-digit)
ca_bg = ca_bg.copy()
acs_df_top10tract = acs_df_top10tract.copy()

# 1) Filter CA block groups to only those in your top-10-tract list
bg_top10 = ca_bg[ca_bg["GEOID"].isin(acs_df_top10tract["GEOID_bg"])].copy()

# 2) Attach demand_weight (merge on tract)
bg_top10 = bg_top10.merge(
    acs_df_top10tract[["GEOID_bg", "demand_weight"]].drop_duplicates("GEOID_bg"),
    left_on="GEOID",
    right_on="GEOID_bg",
    how="left"
).drop(columns=["GEOID_bg"])

In [10]:
bg_top10["centroid"] = bg_top10.geometry.centroid
bg_top10.head()

/var/folders/6g/p95w0f2s53qg_wb80ht4sh4c0000gn/T/ipykernel_22972/633934108.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bg_top10["centroid"] = bg_top10.geometry.centroid


,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,GEOID,GEOIDFQ,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,demand_weight,centroid
0,06,075,026001,2,060750260012,1500000US060750260012,Block Group 2,G5030,S,99140,0,+37.7243172,-122.4318191,"POLYGON ((-122.43443 37.72518, -122.43398 37.7...",188258488,POINT (-122.43182 37.72432)
1,06,075,042700,3,060750427003,1500000US060750427003,Block Group 3,G5030,S,155310,0,+37.7822153,-122.4907785,"POLYGON ((-122.49349 37.7835, -122.49251 37.78...",240884126,POINT (-122.49078 37.78222)
2,06,075,042700,1,060750427001,1500000US060750427001,Block Group 1,G5030,S,117306,0,+37.7838600,-122.4855459,"POLYGON ((-122.48729 37.78565, -122.48622 37.7...",231253728,POINT (-122.48555 37.78386)
3,06,075,042700,2,060750427002,1500000US060750427002,Block Group 2,G5030,S,136780,0,+37.7809706,-122.4874766,"POLYGON ((-122.4913 37.78173, -122.49023 37.78...",146231600,POINT (-122.48748 37.78097)
4,06,075,026100,4,060750261004,1500000US060750261004,Block Group 4,G5030,S,268958,0,+37.7214918,-122.4439587,"POLYGON ((-122.44852 37.71892, -122.44843 37.7...",266964435,POINT (-122.44396 37.72149)


# Getting Muni stops and tract shapefile ready

In [12]:
gdf_stops = gpd.read_file("Muni_Stops_20251214.geojson")
gdf_stops.head()


,:id,:version,:created_at,:updated_at,objectid,stopname,trapezestopabbr,rucusstopabbr,stopid,latitude,...,signupid,supervisor_district,data_as_of,data_loaded_at,:@computed_region_qgnn_b9vv,:@computed_region_26cr_cadq,:@computed_region_ajp5_b2md,:@computed_region_jwn9_ihcz,:@computed_region_6qbp_sg9q,geometry
0,row-9cvw_h2kd.tcxs,rv-cy55-vqj3~d3qm,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,104675,Sansome St&Jackson St SW-FS/BZ,SANSJACK,None,7863,37.796527,...,150,3,2025-11-12 11:26:45,2025-11-15 10:20:22,6,3,6,106,106,POINT (-122.40191 37.79653)
1,row-29a3-x5ji_4ake,rv-u3ae-d6m9_twv3,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,103752,Geary Blvd&Stanyan Blvd SW-NS/BZ,GEARSTA1,GEARSTAN,4764,37.781259999999996,...,150,2,2025-11-12 11:26:45,2025-11-15 10:20:22,8,4,18,12,12,POINT (-122.45643 37.78126)
2,row-rrhz.mn7g-zn3u,rv-ja5g-8meq_b8q7,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102618,Duncan St&Cameo Way SW-NS/SB,DUNCCAM0,DUNCCAMO,4454,37.74519,...,150,8,2025-11-12 11:26:45,2025-11-15 10:20:22,9,5,22,57,57,POINT (-122.44304 37.74519)
3,row-za7s_wfjt-tgpe,rv-pzq4-8s86-dxwn,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102419,Arguello Blvd&California St SE-NS,ARGLCAL1,ARGLCALI,3643,37.785596,...,150,2,2025-11-12 11:26:45,2025-11-15 10:20:22,8,6,31,11,11,POINT (-122.45909 37.7856)
4,row-rhim.uk66.t7ci,rv-dvnb-zuu4-7imh,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,105140,Ocean Ave&Cayuga Ave NE-NS/BZ,OCENCAY0,OCENCAYU,5782,37.72367,...,150,11,2025-11-12 11:26:45,2025-11-15 10:20:22,9,1,28,94,94,POINT (-122.43867 37.72367)


In [14]:
import geopandas as gpd

shp_path = "tl_2023_06_tract/tl_2023_06_tract.shp"
ca_tracts = gpd.read_file(shp_path)

print("Total CA tracts:", len(ca_tracts))


Total CA tracts: 9129


In [15]:
# 1) Select SF tracts
sf_tracts = ca_tracts[ca_tracts["COUNTYFP"] == "075"].copy()

# 2) Select top 10 tracts
top10tract = top10tract.copy()
top10tract["tractce"] = top10tract["tractce"].astype(str).str.zfill(6)

sf_top10_tracts = sf_tracts[
    sf_tracts["TRACTCE"].isin(top10tract["tractce"])
].copy()

# Sanity check
print("SF tracts:", len(sf_tracts))
print("Top 10 tracts:", len(sf_top10_tracts))

sf_top10_tracts.head()

SF tracts: 244
Top 10 tracts: 10


,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
3018,06,075,032802,06075032802,1400000US06075032802,328.02,Census Tract 328.02,G5020,S,548900,97204,+37.7530955,-122.4812305,"POLYGON ((-122.48625 37.75568, -122.48518 37.7..."
3692,06,075,026001,06075026001,1400000US06075026001,260.01,Census Tract 260.01,G5020,S,407294,0,+37.7235502,-122.4322434,"POLYGON ((-122.43647 37.72246, -122.43604 37.7..."
4576,06,075,026100,06075026100,1400000US06075026100,261,Census Tract 261,G5020,S,751208,0,+37.7196238,-122.4431600,"POLYGON ((-122.44869 37.7176, -122.44867 37.71..."
6668,06,075,012001,06075012001,1400000US06075012001,120.01,Census Tract 120.01,G5020,S,62922,0,+37.7884288,-122.4186288,"POLYGON ((-122.42208 37.78847, -122.42185 37.7..."
6705,06,075,011001,06075011001,1400000US06075011001,110.01,Census Tract 110.01,G5020,S,118009,0,+37.7934678,-122.4196321,"POLYGON ((-122.42314 37.79392, -122.42297 37.7..."


# Transit stops in SF tracts

In [16]:
import geopandas as gpd

# 0) Make sure both layers use the same CRS
gdf_stops = gdf_stops.to_crs(sf_top10_tracts.crs)

# 1) Spatial join: keep only stops within top10 tracts, and bring TRACTCE over
stops_in_top10 = gpd.sjoin(
    gdf_stops,
    sf_top10_tracts[["TRACTCE", "geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"])

# stops_in_top10 now contains all original stop columns + TRACTCE
stops_in_top10.head()


,:id,:version,:created_at,:updated_at,objectid,stopname,trapezestopabbr,rucusstopabbr,stopid,latitude,...,supervisor_district,data_as_of,data_loaded_at,:@computed_region_qgnn_b9vv,:@computed_region_26cr_cadq,:@computed_region_ajp5_b2md,:@computed_region_jwn9_ihcz,:@computed_region_6qbp_sg9q,geometry,TRACTCE
49,row-62xi.iakd_wcsr,rv-p3au~djhc_7e6x,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102534,Brazil Ave&Mission St E-NS/PS,BRZLMISS,BRZLMISS,3761,37.724717,...,11,2025-11-12 11:26:45,2025-11-15 10:20:22,9,1,7,90,90,POINT (-122.43465 37.72472),026001
127,row-kwmy~56qq_e3hg,rv-mifw.x8u8.vfji,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102414,23rd Ave&Noriega St NW-NS/PS,23AVNOR0,23AVNORI,3447,37.75414,...,4,2025-11-12 11:26:45,2025-11-15 10:20:22,10,7,35,39,39,POINT (-122.48086 37.75414),032802
167,row-fq3c~xiji.37hh,rv-6s6f-6bn8.tysx,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102261,33rd Ave&Geary Blvd NE-FS/SB,33AVGEA1,33AVGEAR,3557,37.779815,...,1,2025-11-12 11:26:45,2025-11-15 10:20:22,8,4,29,8,8,POINT (-122.49323 37.77982),042700
228,row-d2fh-yhfj-g68e,rv-bfnu~n4sr_ka2r,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,104578,Balboa Park BART Station SW-MB/BZ,GNVABAR,GNVABART,4805,37.720952,...,11,2025-11-12 11:26:45,2025-11-15 10:20:22,9,1,28,80,80,POINT (-122.44738 37.72095),026100
238,row-9fx2-iirt~6jzd,rv-ku7w.2k22~9pwe,2025-11-15 18:20:31.567000+00:00,2025-11-15 18:20:33.746000+00:00,102254,32nd Ave&Clement St NW-NS/PS,32AVCLM0,32AVCLMT,3549,37.781802,...,1,2025-11-12 11:26:45,2025-11-15 10:20:22,8,4,29,8,8,POINT (-122.49242 37.7818),042700


In [17]:
# Count number of stops per tract
tract_stop_counts = (
    stops_in_top10
    .groupby("TRACTCE")
    .size()
    .reset_index(name="num_stops")
    .sort_values("num_stops", ascending=False)
    .reset_index(drop=True)
)

# Add ranking (1 = most stops)
tract_stop_counts["rank"] = tract_stop_counts["num_stops"].rank(
    method="dense", ascending=False
).astype(int)

tract_stop_counts


,TRACTCE,num_stops,rank
0,026100,26,1
1,042700,20,2
2,011101,17,3
3,026001,17,3
4,032802,13,4
5,032902,9,5
6,012001,8,6
7,011102,7,7
8,011001,6,8
9,012002,2,9


# Mid block points

Reading mid block points

In [18]:
import geopandas as gpd

# 1) Load the new Mid-Block Points GeoJSON
# Make sure the file path matches where your file is stored
gdf_midblocks = gpd.read_file("San_Francisco_Mid-Block_Points_20251215.geojson")

# 2) Make sure both layers use the same CRS
# We project the new mid-block points to match the tracts CRS
gdf_midblocks = gdf_midblocks.to_crs(sf_top10_tracts.crs)

# 3) Spatial join: keep only points within top10 tracts, and bring TRACTCE over
midblocks_in_top10 = gpd.sjoin(
    gdf_midblocks,
    sf_top10_tracts[["TRACTCE", "geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"])

# midblocks_in_top10 now contains all original columns + TRACTCE
midblocks_in_top10.head()

,:id,:version,:created_at,:updated_at,cnn,parity,mid_block_address,longitude,latitude,supervisor_district,analysis_neighborhood,data_as_of,data_loaded_at,geometry,TRACTCE
212,row-2gxr-t68q.hacf,rv-xuff~f6v4-nars,2025-12-14 18:06:18.638000+00:00,2025-12-14 18:06:18.638000+00:00,12328000,Even,1000 Block of SUTTER ST,-122.417657212,37.788126304,3,Nob Hill,2025-12-14 04:30:00,2025-12-14 10:06:02.788,POINT (-122.41766 37.78813),012001
266,row-d9xq_ezeq~3ive,rv-jjhk.5kvi~rtzx,2025-12-14 18:06:18.638000+00:00,2025-12-14 18:06:18.638000+00:00,11798000,Even,0 Block of SHAWNEE AVE,-122.445693904,37.717959404,11,Outer Mission,2025-12-14 04:30:00,2025-12-14 10:06:02.788,POINT (-122.44569 37.71796),026100
271,row-nbc6.2qcg.qqct,rv-w5um_vyqb_6r27,2025-12-14 18:06:18.638000+00:00,2025-12-14 18:06:18.638000+00:00,11761000,Even,200 Block of SENECA AVE,-122.443164705,37.72061922,11,Outer Mission,2025-12-14 04:30:00,2025-12-14 10:06:02.788,POINT (-122.44316 37.72062),026100
305,row-emw8_gahs_xzjt,rv-amh8~92vj.v43m,2025-12-14 18:06:18.638000+00:00,2025-12-14 18:06:18.638000+00:00,11457000,Even,2200 Block of SAN JOSE AVE,-122.446289445,37.721206733,11,Outer Mission,2025-12-14 04:30:00,2025-12-14 10:06:02.788,POINT (-122.44629 37.72121),026100
372,row-7ik5_7zs4.p582,rv-t927_n7xw_8pzr,2025-12-14 18:06:18.638000+00:00,2025-12-14 18:06:18.638000+00:00,10861000,Even,1600 Block of QUINTARA ST,-122.483051863,37.748392918,4,Sunset/Parkside,2025-12-14 04:30:00,2025-12-14 10:06:02.788,POINT (-122.48305 37.74839),032802


In [19]:
# Count number of midblocks per tract
tract_midblock_counts = (
    midblocks_in_top10  # Assuming your input dataframe is named this
    .groupby("TRACTCE")
    .size()
    .reset_index(name="num_midblocks")
    .sort_values("num_midblocks", ascending=False)
    .reset_index(drop=True)
)

# Add ranking (1 = most midblocks)
tract_midblock_counts["rank"] = tract_midblock_counts["num_midblocks"].rank(
    method="dense", ascending=False
).astype(int)

tract_midblock_counts

,TRACTCE,num_midblocks,rank
0,026100,244,1
1,032802,111,2
2,032902,108,3
3,026001,106,4
4,042700,87,5
5,011101,47,6
6,011102,44,7
7,011001,36,8
8,012002,26,9
9,012001,25,10


# Biz in SF tracts

In [20]:
import geopandas as gpd
gdf_biz = gpd.read_file("Registered_Business_Locations_-_San_Francisco_20251214.geojson")
gdf_biz.head()

,uniqueid,certificate_number,ttxid,ownership_name,dba_name,full_business_address,city,state,business_zip,dba_start_date,...,:id,:version,:created_at,:updated_at,:@computed_region_6qbp_sg9q,:@computed_region_qgnn_b9vv,:@computed_region_26cr_cadq,:@computed_region_ajp5_b2md,:@computed_region_jwn9_ihcz,geometry
0,1012461-11-141-1006177,1006177,1012461-11-141,Mcgill Linda,Air West Filtration,89 Brentwood Dr,San Fafael,CA,94901,2014-11-01,...,row-tg7p-fcjr_mdym,rv-xstg.r9cj-e2zh,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,None,None,None,None,None,POINT (-122.48173 37.99117)
1,0407659-01-999-0407659,0407659,0407659-01-999,Fu Kyaw L,Fu Kyaw L,4043 Timberline Dr,San Jose,CA,95121,2007-01-01,...,row-r5tp_aapn-7cmf,rv-ynid~aitv~at5f,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,None,None,None,None,None,POINT (-121.78081 37.3057)
2,0444525-02-999-0444525,0444525,0444525-02-999,Jones Joseph C,Jj Electric,1008 Sterling St,Vallejo,CA,94591,2009-11-03,...,row-vgu9~9xqu.32dq,rv-enqz_gsmc.fmuv,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,None,None,None,None,None,None
3,1050673-01-161-1023892,1023892,1050673-01-161,Fortifire Inc,Fortifire Inc,46560 Fremont Blvd #119,Fremont,CA,94538,2016-01-01,...,row-fcdg-xz83_spy6,rv-9xhc-3v97_iivy,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,None,None,None,None,None,POINT (-121.94497 37.4836)
4,0480213-01-999-0480213,0480213,0480213-01-999,Deleon Marco A,Marco Deleon Plumbing,273 88th St 8,Daly City,CA,94015,2013-06-07,...,row-sbu7.r8ga_ata3,rv-kah9.af8m-7wxb,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,None,None,None,None,None,POINT (-122.47404 37.69217)


In [21]:
gdf_biz_SF = gdf_biz[gdf_biz['city'] == 'San Francisco']

In [22]:
import geopandas as gpd

# 0) Make sure both layers use the same CRS
gdf_biz_SF = gdf_biz_SF.to_crs(sf_top10_tracts.crs)

# 1) Spatial join: keep only business POIs within top10 tracts, and bring TRACTCE over
biz_in_top10 = gpd.sjoin(
    gdf_biz_SF,
    sf_top10_tracts[["TRACTCE", "geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"])

# biz_in_top10 now contains all original business columns + TRACTCE
biz_in_top10.head()


,uniqueid,certificate_number,ttxid,ownership_name,dba_name,full_business_address,city,state,business_zip,dba_start_date,...,:version,:created_at,:updated_at,:@computed_region_6qbp_sg9q,:@computed_region_qgnn_b9vv,:@computed_region_26cr_cadq,:@computed_region_ajp5_b2md,:@computed_region_jwn9_ihcz,geometry,TRACTCE
632,1277319-05-211-1124285,1124285,1277319-05-211,Katherine Fiordalis,Katherine Fiordalis,6705 California St Apt 4,San Francisco,CA,94121,2020-01-10,...,rv-9sxe.zhb3~9nxb,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,8,8,4,29,8,POINT (-122.48958 37.78354),042700
742,1372719-10-241-1163092,1163092,1372719-10-241,Alvares Ramos Inc,Restaurante Familiar,4499 Mission St,San Francisco,CA,94112,2024-02-22,...,rv-2phf.48vi-nffs,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,90,9,1,7,90,POINT (-122.43329 37.72625),026001
779,1334023-05-231-1142068,1142068,1334023-05-231,John Oliver Por,Por Rentals,2379 Alemany Blvd,San Francisco,CA,94112,2022-12-31,...,rv-jwk7_a63b.vzjh,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,80,9,1,28,80,POINT (-122.44414 37.71609),026100
847,1352710-02-241-1155086,1155086,1352710-02-241,Nadezhda Kiryukhina,Elegansee,451 27th Ave,San Francisco,CA,94121,2024-02-11,...,rv-ywsg.ih8u~65gi,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,8,8,4,29,8,POINT (-122.48729 37.78095),042700
893,0478473-02-001-0478473,0478473,0478473-02-001,Wong Winchelle,Winchelles Child Care,1770 32nd Ave,San Francisco,CA,94122,2013-05-01,...,rv-e8e7_vvxs-6gn4,2025-12-14 12:24:21.161000+00:00,2025-12-14 12:31:12.863000+00:00,39,10,7,35,39,POINT (-122.49013 37.75421),032902


In [23]:
# Count number of businesses per tract
tract_biz_counts = (
    biz_in_top10
    .groupby("TRACTCE")
    .size()
    .reset_index(name="num_biz")
    .sort_values("num_biz", ascending=False)
    .reset_index(drop=True)
)

# Add ranking (1 = most businesses)
tract_biz_counts["rank"] = tract_biz_counts["num_biz"].rank(
    method="dense", ascending=False
).astype(int)

tract_biz_counts

,TRACTCE,num_biz,rank
0,042700,1289,1
1,026100,1220,2
2,011101,1046,3
3,026001,1029,4
4,032802,1004,5
5,011102,997,6
6,032902,787,7
7,012001,763,8
8,011001,758,9
9,012002,630,10


# Proximity score calculation

In [24]:
import geopandas as gpd

# --------------------------
# Params
# --------------------------
R_M = 300
EPS_M = 0.001                     # treat as meters; if you meant 0.001 miles use: 0.001*1609.344
CRS_METERS = "EPSG:26910"         # UTM 10N (SF)

# --------------------------
# 0) Project + add candidate id
# --------------------------
mid = midblocks_in_top10.to_crs(CRS_METERS).copy()
stops = stops_in_top10.to_crs(CRS_METERS).copy()
biz = biz_in_top10.to_crs(CRS_METERS).copy()

mid = mid.reset_index(drop=True)
mid["cand_id"] = mid.index

mid = mid[["cand_id", "TRACTCE", "geometry"]]
stops = stops[["TRACTCE", "geometry"]]
biz = biz[["TRACTCE", "geometry"]]

# --------------------------
# 1) Nearest stop distance per candidate (within tract)
# --------------------------
out_stop = []
for tract, mid_t in mid.groupby("TRACTCE"):
    stops_t = stops[stops["TRACTCE"] == tract]
    if len(stops_t) == 0:
        tmp = mid_t[["cand_id"]].copy()
        tmp["d_t_m"] = float("nan")
    else:
        tmp = gpd.sjoin_nearest(
            mid_t,
            stops_t,
            how="left",
            distance_col="d_t_m"
        )[["cand_id", "d_t_m"]]
        tmp = tmp.groupby("cand_id", as_index=False)["d_t_m"].min()
    out_stop.append(tmp)

d_stop = gpd.pd.concat(out_stop, ignore_index=True)

# --------------------------
# 2) Nearest business distance per candidate (within tract)
# --------------------------
out_biz = []
for tract, mid_t in mid.groupby("TRACTCE"):
    biz_t = biz[biz["TRACTCE"] == tract]
    if len(biz_t) == 0:
        tmp = mid_t[["cand_id"]].copy()
        tmp["d_b_m"] = float("nan")
    else:
        tmp = gpd.sjoin_nearest(
            mid_t,
            biz_t,
            how="left",
            distance_col="d_b_m"
        )[["cand_id", "d_b_m"]]
        tmp = tmp.groupby("cand_id", as_index=False)["d_b_m"].min()
    out_biz.append(tmp)

d_biz = gpd.pd.concat(out_biz, ignore_index=True)

# --------------------------
# 3) Merge distances back + scores
# --------------------------
candidates = mid.merge(d_stop, on="cand_id", how="left").merge(d_biz, on="cand_id", how="left")

candidates["St"] = (R_M - candidates["d_t_m"] + EPS_M) / R_M
candidates["Sb"] = (R_M - candidates["d_b_m"] + EPS_M) / R_M

# Optional: keep scores in [0, 1]
candidates["St"] = candidates["St"].clip(0, 1)
candidates["Sb"] = candidates["Sb"].clip(0, 1)

In [25]:
candidates

,cand_id,TRACTCE,geometry,d_t_m,d_b_m,St,Sb
0,0,012001,POINT (551275.111 4182467.286),66.716141,17.933232,0.777616,0.940226
1,1,026100,POINT (548852.583 4174667.353),206.709250,14.573696,0.310973,0.951424
2,2,026100,POINT (549073.735 4174963.777),190.861065,34.499017,0.363800,0.885007
3,3,026100,POINT (548797.965 4175027.328),6.882278,40.596588,0.977062,0.864681
4,4,032802,POINT (545541.451 4178025.066),35.104625,160.497958,0.882988,0.465010
...,...,...,...,...,...,...,...
829,829,042700,POINT (545112.255 4181940.817),68.530931,20.577203,0.771567,0.931413
830,830,011102,POINT (551080.237 4182743.75),61.338342,18.626568,0.795542,0.937915
831,831,026001,POINT (549845.281 4175404.142),30.656553,16.899291,0.897815,0.943672
832,832,011101,POINT (551414.607 4182837.288),130.971729,6.359822,0.563431,0.978804


# Suitability score calculation

In [26]:
import numpy as np

# Work on a copy (optional but safe)
candidates = candidates.copy()

# Per-tract min / max
st_min = candidates.groupby("TRACTCE")["St"].transform("min")
st_max = candidates.groupby("TRACTCE")["St"].transform("max")
sb_min = candidates.groupby("TRACTCE")["Sb"].transform("min")
sb_max = candidates.groupby("TRACTCE")["Sb"].transform("max")

# Avoid divide-by-zero
st_range = (st_max - st_min).replace(0, np.nan)
sb_range = (sb_max - sb_min).replace(0, np.nan)

# Normalized scores within each tract
candidates["St_norm"] = (candidates["St"] - st_min) / st_range
candidates["Sb_norm"] = (candidates["Sb"] - sb_min) / sb_range

# If all candidates in a tract have identical St or Sb
candidates[["St_norm", "Sb_norm"]] = (
    candidates[["St_norm", "Sb_norm"]].fillna(0)
)

# Suitability score
candidates["SS"] = (
    0.5 * candidates["St_norm"] +
    0.5 * candidates["Sb_norm"]
)

In [27]:
candidates

,cand_id,TRACTCE,geometry,d_t_m,d_b_m,St,Sb,St_norm,Sb_norm,SS
0,0,012001,POINT (551275.111 4182467.286),66.716141,17.933232,0.777616,0.940226,0.499482,0.412427,0.455955
1,1,026100,POINT (548852.583 4174667.353),206.709250,14.573696,0.310973,0.951424,0.314367,0.866249,0.590308
2,2,026100,POINT (549073.735 4174963.777),190.861065,34.499017,0.363800,0.885007,0.367770,0.657946,0.512858
3,3,026100,POINT (548797.965 4175027.328),6.882278,40.596588,0.977062,0.864681,0.987726,0.594201,0.790964
4,4,032802,POINT (545541.451 4178025.066),35.104625,160.497958,0.882988,0.465010,1.000000,0.473978,0.736989
...,...,...,...,...,...,...,...,...,...,...
829,829,042700,POINT (545112.255 4181940.817),68.530931,20.577203,0.771567,0.931413,0.718982,0.699971,0.709477
830,830,011102,POINT (551080.237 4182743.75),61.338342,18.626568,0.795542,0.937915,0.635721,0.508957,0.572339
831,831,026001,POINT (549845.281 4175404.142),30.656553,16.899291,0.897815,0.943672,0.883891,0.788696,0.836293
832,832,011101,POINT (551414.607 4182837.288),130.971729,6.359822,0.563431,0.978804,0.099764,0.859301,0.479533


# Market share calculation

In [28]:
import numpy as np
import geopandas as gpd

def huff_market_share(
    candidates: gpd.GeoDataFrame,
    bg_top10: gpd.GeoDataFrame,
    tract_col: str = "TRACTCE",
    ss_col: str = "SS",
    demand_col: str = "demand_weight",
    bg_centroid_col: str = "centroid",
    beta: float = 1.5,          # distance decay exponent (try 1.0~2.0)
    alpha: float = 1.0,         # attractiveness exponent on SS (try 1.0~2.0)
    eps_m: float = 1.0,         # epsilon in meters to avoid /0
    crs_m: str = "EPSG:26910"   # meters CRS for SF
):
    # ---- 0) Make working copies in a meters CRS ----
    cand = candidates.copy()
    bg = bg_top10.copy()

    # Candidates: ensure point geometry in meters CRS
    cand = cand.to_crs(crs_m)

    # BGs: use centroid column as geometry
    bg = bg.set_geometry(bg_centroid_col)
    bg = bg.to_crs(crs_m)

    # Keys as strings (safe)
    cand[tract_col] = cand[tract_col].astype(str)
    bg[tract_col] = bg[tract_col].astype(str)

    # Prepare output
    cand["market_share"] = np.nan

    # ---- 1) Compute market share tract-by-tract ----
    for tract, cand_t in cand.groupby(tract_col):
        bg_t = bg[bg[tract_col] == tract]

        # If no demand points in tract, market share = 0
        if len(bg_t) == 0:
            cand.loc[cand_t.index, "market_share"] = 0.0
            continue

        # If only 1 candidate, it captures all demand in that tract
        if len(cand_t) == 1:
            total_demand = bg_t[demand_col].sum()
            cand.loc[cand_t.index, "market_share"] = float(total_demand)
            continue

        # Coordinates (meters)
        cand_xy = np.column_stack([cand_t.geometry.x.values, cand_t.geometry.y.values])
        bg_xy = np.column_stack([bg_t.geometry.x.values, bg_t.geometry.y.values])

        # Distances: shape (n_bg, n_cand)
        # d_ij = distance from BG i to candidate j
        dx = bg_xy[:, None, 0] - cand_xy[None, :, 0]
        dy = bg_xy[:, None, 1] - cand_xy[None, :, 1]
        d = np.sqrt(dx*dx + dy*dy)

        # Attractiveness A_j = SS^alpha (clip to avoid negative / nan)
        A = np.clip(cand_t[ss_col].astype(float).values, 0, None) ** alpha  # shape (n_cand,)

        # Distance decay f(d) = 1/(d+eps)^beta
        decay = 1.0 / np.power(d + eps_m, beta)  # shape (n_bg, n_cand)

        # Numerator for each BG-candidate
        numer = decay * A[None, :]  # shape (n_bg, n_cand)

        # Probabilities P_ij
        denom = numer.sum(axis=1, keepdims=True)  # shape (n_bg, 1)

        # If denom==0 for any BG (e.g., all SS=0), split evenly among candidates
        zero = (denom.squeeze() == 0)
        P = np.zeros_like(numer)
        if np.any(zero):
            P[zero, :] = 1.0 / numer.shape[1]
        if np.any(~zero):
            P[~zero, :] = numer[~zero, :] / denom[~zero, :]

        # Market share per candidate: sum_i demand_i * P_ij
        demand = bg_t[demand_col].astype(float).values  # shape (n_bg,)
        ms = (demand[:, None] * P).sum(axis=0)          # shape (n_cand,)

        cand.loc[cand_t.index, "market_share"] = ms

    # ---- 2) Best candidate per tract (max market_share) ----
    best_idx = cand.groupby(tract_col)["market_share"].idxmax()
    best_per_tract = cand.loc[best_idx].copy().sort_values(tract_col).reset_index(drop=True)

    # Return in original CRS (optional). If you prefer meters CRS, remove these lines.
    cand = cand.to_crs(candidates.crs)
    best_per_tract = best_per_tract.to_crs(candidates.crs)

    return cand, best_per_tract


# =========================
# Run it
# =========================
candidates_ms, best_candidate_per_tract = huff_market_share(
    candidates=candidates,
    bg_top10=bg_top10,
    beta=1.5,     # tune
    alpha=1.0,    # tune
    eps_m=1.0
)

# candidates_ms: all candidates with market_share
# best_candidate_per_tract: one best candidate per tract
candidates_ms.head()


,cand_id,TRACTCE,geometry,d_t_m,d_b_m,St,Sb,St_norm,Sb_norm,SS,market_share
0,0,012001,POINT (551275.111 4182467.286),66.716141,17.933232,0.777616,0.940226,0.499482,0.412427,0.455955,2.556440e+06
1,1,026100,POINT (548852.583 4174667.353),206.709250,14.573696,0.310973,0.951424,0.314367,0.866249,0.590308,4.102381e+06
2,2,026100,POINT (549073.735 4174963.777),190.861065,34.499017,0.363800,0.885007,0.367770,0.657946,0.512858,3.925429e+06
3,3,026100,POINT (548797.965 4175027.328),6.882278,40.596588,0.977062,0.864681,0.987726,0.594201,0.790964,3.164880e+06
4,4,032802,POINT (545541.451 4178025.066),35.104625,160.497958,0.882988,0.465010,1.000000,0.473978,0.736989,2.254451e+06


In [29]:
best_candidate_per_tract

,cand_id,TRACTCE,geometry,d_t_m,d_b_m,St,Sb,St_norm,Sb_norm,SS,market_share
0,120,011001,POINT (550955.038 4183084.764),14.371729,5.122065,0.952098,0.982930,0.921985,0.932636,0.927311,2.344730e+07
1,26,011101,POINT (551293.834 4182836.921),38.731735,23.653836,0.870898,0.921157,0.744940,0.275939,0.510440,2.990063e+07
2,94,011102,POINT (551154.695 4182707.75),46.328791,26.840546,0.845574,0.910535,0.729528,0.242356,0.485942,1.309804e+08
3,457,012001,POINT (551121.851 4182494.677),86.162206,18.930279,0.712796,0.936902,0.303052,0.369928,0.336490,3.108724e+07
4,761,012002,POINT (551362.793 4182422.779),295.002203,8.805256,0.016663,0.970652,0.016867,0.871042,0.443955,3.860785e+07
5,92,026001,POINT (550074.29 4175378.161),116.434025,26.601517,0.611890,0.911332,0.451031,0.634557,0.542794,1.070455e+08
6,251,026100,POINT (548812.162 4174734.925),162.850012,21.450576,0.457170,0.928501,0.462160,0.794356,0.628258,2.447466e+07
7,599,032802,POINT (545941.078 4178562.507),159.187912,22.041275,0.469377,0.926532,0.531578,0.944401,0.737989,2.909972e+07
8,159,032902,POINT (544878.347 4178705.323),143.338483,23.987525,0.522208,0.920045,0.537134,0.671688,0.604411,4.473498e+07
9,153,042700,POINT (545299.369 4181964.727),26.284539,18.203214,0.912388,0.939326,0.906338,0.736936,0.821637,1.059138e+08


Return lat/lon of best sites

In [30]:
import pandas as pd

# Ensure CRS is lat/lon
best_candidate_per_tract = best_candidate_per_tract.to_crs("EPSG:4326")

# Extract coordinates
df_latlon = pd.DataFrame({
    "TRACTCE": best_candidate_per_tract["TRACTCE"],
    "lon": best_candidate_per_tract.geometry.x,
    "lat": best_candidate_per_tract.geometry.y
})

df_latlon

,TRACTCE,lon,lat
0,011001,-122.421249,37.793709
1,011101,-122.417418,37.791457
2,011102,-122.419008,37.790300
3,012001,-122.419396,37.788382
4,012002,-122.416665,37.787720
5,026001,-122.431783,37.724300
6,026100,-122.446148,37.718571
7,032802,-122.478482,37.753217
8,032902,-122.490537,37.754557
9,042700,-122.485554,37.783913


In [31]:
df_latlon.to_csv('best_candidate_per_tract_lat_lon.csv')

In [32]:
best_candidate_per_tract.to_file(
    "best_candidate_per_tract.geojson",
    driver="GeoJSON"
)

In [34]:
best_candidate_per_tract.to_file(
    "best_candidate_per_tract/best_candidate_per_tract.shp"
)


/var/folders/6g/p95w0f2s53qg_wb80ht4sh4c0000gn/T/ipykernel_22972/2882870334.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  best_candidate_per_tract.to_file(
/Users/yijinwang/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'market_share' to 'market_sha'
  ogr_write(
/Users/yijinwang/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 130980399.79780236 of field market_sha of feature 2 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
/Users/yijinwang/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 107045493.51718399 of field market_sha of feature 5 not successfully written. Possibly due to too larger number with respect to field width
  ogr_write(
/Users/yijinwang/Library/Python/3.9/lib/python/site-packages/pyogrio/raw.py:723: RuntimeWarning: